In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt

from adaptive_latents import datasets, proSVD, Pipeline, CenteringEstimator, StreamingKalmanFilter, Bubblewrap, sjPCA, mmICA, ArrayWithTime, plotting_functions, KernelSmoother, VJF
from adaptive_latents.utils import save_to_cache
from tqdm.auto import tqdm
from IPython import display
from adaptive_latents.plotting_functions import plot_flow_fields, AnimationManager, plot_history_with_tail
import adaptive_latents
import importlib

from adaptive_latents.predictor import Predictor
from adaptive_latents.regressions import BaseKernelRegressor


In [ ]:

d = datasets.Zong22Dataset()


prosvd_k = 8

# @save_to_cache("parallel_compare")
def f():
    p = Pipeline([CenteringEstimator(), KernelSmoother(tau=2 * .68 / d.neural_data.dt), proSVD(k=prosvd_k)])

    dim_red_methods = [Pipeline(), sjPCA(), mmICA()]
    # predictors = [StreamingKalmanFilter(log_level=2, check_dt=True, n_steps_to_predict=1, steps_between_refits=50) for _ in dim_red_methods]
    predictors = [Bubblewrap(log_level=2, check_dt=True, n_steps_to_predict=1) for _ in dim_red_methods]
    # predictors = [VJF(log_level=2, check_dt=True, n_steps_to_predict=1) for _ in dim_red_methods]

    regs = [BaseKernelRegressor(maxlen=10000, length_scale=0.1725) for _ in dim_red_methods]

    outputs = [[] for _ in dim_red_methods]

    pbar = tqdm(total=round(d.neural_data.t.max(),2))
    for data in p.streaming_run_on(d.neural_data):

        metrics = []
        in_space_data = []
        for dim_red_method, predictor, output_accumulator in zip(dim_red_methods, predictors, outputs):
            in_space_datum = dim_red_method.partial_fit_transform(data)
            in_space_datum = in_space_datum[:,:4]
            in_space_data.append(in_space_datum)
            output_accumulator.append(in_space_datum)

            mse = ((in_space_datum - predictor.predict(1)) ** 2).mean()
            neg_log_pred_p = -predictor.unevaluated_log_pred_p(1)(in_space_datum)
            metrics.append(neg_log_pred_p)
            predictor.partial_fit_transform(in_space_datum)

        best_regressor = np.argmin(metrics)
        for i, (reg, in_space_datum) in enumerate(zip(regs, in_space_data)):
            reg.observe(in_space_datum, np.array([i == best_regressor]))

        pbar.update(round(data.t,2) - pbar.n)


    outputs = [ArrayWithTime.from_list(o, drop_early_nans=True, squeeze_type='to_2d') for o in outputs]
    return outputs, dim_red_methods, regs, predictors

outputs, dim_red_methods, regs, predictors = f()

labels = ['prosvd','sjpca','mmica']



In [ ]:
predictor_lpps = [ArrayWithTime.from_list(p.log['log_pred_p'], squeeze_type='squeeze') for p in predictors]
predictor_lpps = ArrayWithTime(np.column_stack(predictor_lpps), predictor_lpps[0].t)

In [ ]:
chosen = [0]
combined = []
for i in range(len(predictor_lpps)):
    combined.append(predictor_lpps[i, chosen[-1]])
    best = np.argmax(predictor_lpps[i,:])
    chosen.append(best)
combined = ArrayWithTime(combined, predictor_lpps.t)


In [ ]:
%matplotlib inline

fig, ax = plt.subplots(constrained_layout=True, figsize=(10,4))
for label, lpp in zip(labels, predictor_lpps.T):
    ax.plot(lpp.t, lpp, '.-', label=label)
    print(f'{label} mse_whole={np.nanmean(lpp):.2f} mse[600:]={np.nanmean(lpp.slice_by_time(slice(300,None))):.2f}')

ax.plot(combined.t, combined, '.-k', label='best')
print(f'combined mse_whole={np.nanmean(combined):.2f} mse[600:]={np.nanmean(combined.slice_by_time(slice(300,None))):.2f}')

ax.legend()

ax.set_xlim([425, 439])
ax.set_ylim([-12, 3])
fig.savefig('/home/jgould/Downloads/parallel_compare_lpp_zoom.svg')
